# Diffusion run metrics — comparison dashboard

Compares the diffusion runs on the residuals we've been tracking, plus marginal KL.
Re-run top to bottom; generation (Section 2) is the only slow part and is cached in
`GEN`, so the plot cells below it are cheap to re-run.

**Residual legend**
- **R1 — under-dispersion**: synthetic windows for a fixed config are less spread out
  than real ones (`div_ratio`, `std_ratio_med`; →1 is best).
- **R2 — over-regular days**: generated weeks repeat a near-identical daily shape
  (`ac24_gap` = |synth−real lag-24 autocorrelation|; ↓ best).
- **R3 — marginal shape**: per-KPI value-distribution mismatch (`ks_med`, `kl_med`; ↓ best).

**Cross-run comparability** — runs 3/4 train in raw `[0,1]`, runs `*_yj` in
Yeo-Johnson→`[0,1]`. KS is invariant under that monotonic transform, so `ks_med` and
the per-KPI KS boxplot are *directly* comparable across runs. `div_ratio`/`std_ratio`
are ratios (≈comparable). **KL and the marginal-histogram overlays are computed in each
run's own training space** — compare a run against *its own* real data (within a column),
not absolute KL values across spaces. For physical-unit marginals use
`run_visual_checks5.ipynb`'s inverse pipeline.

In [ ]:
import getpass
import sys
import json

_USER = getpass.getuser()
sys.path.insert(0, f"/home/{_USER}/app/apps/apps/generator")

import tsgm  # noqa: E402, F401  -- must be imported before keras
import keras  # noqa: E402, F401
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.stats import ks_2samp  # noqa: E402

from genpm.modelling.core.artifacts import load_trained_model  # noqa: E402
from genpm.modelling.core.data import build_calendar_features  # noqa: E402
from genpm.utils.consts import SHARED_DIR_PATH  # noqa: E402

In [ ]:
# ── Runs to compare (label, run_dir, weights_file) ──
RUNS = [
    ("run_3",    "diffusion_run_3",    "ddpm_3.weights.h5"),
    ("run_4",    "diffusion_run_4",    "ddpm_4.weights.h5"),
    ("run_4_yj", "diffusion_run_4_yj", "ddpm_4_yj.weights.h5"),
    ("run_5_yj", "diffusion_run_5_yj", "ddpm_5_yj.weights.h5"),
]
MODEL_RUNS = SHARED_DIR_PATH / "model_runs"
RUN_DIR = {label: run for label, run, _ in RUNS}


def history(label):
    return json.load(open(MODEL_RUNS / RUN_DIR[label] / "training_history.json"))

## 1. Training history

Logged every epoch during training (no model loading needed). `gen_*` are the
eval-callback's generated-sample metrics, dashed lines are each run's real-data
reference (same colour). Watch `gen_diversity` climb toward its real line (R1) and
`gen_ac1` settle near real (not far above → R2).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for label, _, _ in RUNS:
    h = history(label)
    ep = np.arange(1, len(h["loss"]) + 1)
    axes[0].plot(ep, h["loss"], label=label)
    ln, = axes[1].plot(ep, h["gen_diversity"], label=label)
    axes[1].axhline(h["real_diversity"][0], ls="--", color=ln.get_color(), alpha=0.4)
    ln2, = axes[2].plot(ep, h["gen_ac1"], label=label)
    axes[2].axhline(h["real_ac1"][0], ls="--", color=ln2.get_color(), alpha=0.4)

axes[0].set_title("noise-prediction loss (MSE) ↓"); axes[0].set_xlabel("epoch"); axes[0].set_yscale("log")
axes[1].set_title("gen vs real diversity (dashed=real)  R1")
axes[2].set_title("gen vs real lag-1 AC (dashed=real)  R2")
for ax in axes:
    ax.legend(fontsize=8); ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 2. Generate synthetic windows (cached in `GEN`)

For each run: load the model, pick the most common config row, take up to 64 real
windows for it, and generate matched synthetic windows (rebuilding per-timestep
calendar features for the calendar-conditioned runs). Stored as
`GEN[label] = {Xr, Xf, kc}` (real, synth, kpi columns) — all in that run's own scaled
space. **This is the slow cell (~1 min/run).**

In [ ]:
def _tn(x):
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)


def generate_for_run(run, weights):
    R = MODEL_RUNS / run
    model, _, _ = load_trained_model(run_id_path=R, weights_path=R / "models_weights_debug" / weights)
    X = np.load(R / "X_scaled.npy")
    y = np.load(R / "y.npy")
    wa = pd.to_datetime(pd.Series(np.load(R / "window_anchors.npy", allow_pickle=True))).values
    kc = list(np.load(R / "kpi_columns.npy", allow_pickle=True))

    uy, counts = np.unique(y, axis=0, return_counts=True)
    row = uy[int(np.argmax(counts))]                      # most common config
    idx = np.where(np.all(y == row, axis=1))[0]
    idx = np.random.default_rng(0).choice(idx, min(64, len(idx)), replace=False)

    Xr = X[idx]
    yb = np.repeat(row[None], len(idx), 0).astype("float32")
    cal = (build_calendar_features(wa[idx], 168).astype("float32")
           if getattr(model, "cond_dim", 0) > 0 else None)
    Xf = _tn(model.generate(yb, calendar=cal)[0]) if cal is not None else _tn(model.generate(yb)[0])
    return dict(Xr=Xr.astype("float32"), Xf=Xf.astype("float32"), kc=kc)


GEN = {}
for label, run, weights in RUNS:
    print(f"generating {label} ...", flush=True)
    GEN[label] = generate_for_run(run, weights)
print("done — Xf shapes:", {l: GEN[l]["Xf"].shape for l, _, _ in RUNS})

## 3. Scalar metric suite

One row per run. Cross-run-comparable metrics (`ks_med`, ratios, autocorr gaps) sit
alongside within-space ones (`kl_med`, `loss_final`).

In [ ]:
def lag_ac(a, lag):
    x, b = a[:, :-lag], a[:, lag:]
    x = x - x.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    return float(np.nanmean((x * b).sum(1) / (np.sqrt((x ** 2).sum(1) * (b ** 2).sum(1)) + 1e-8)))


def kl_hist(real, fake, bins=50):
    lo = float(min(real.min(), fake.min()))
    hi = float(max(real.max(), fake.max()))
    if hi <= lo:
        return 0.0
    pr, _ = np.histogram(real, bins=bins, range=(lo, hi))
    pf, _ = np.histogram(fake, bins=bins, range=(lo, hi))
    pr = pr + 1.0; pf = pf + 1.0                          # Laplace smoothing
    pr /= pr.sum(); pf /= pf.sum()
    return float(np.sum(pr * np.log(pr / pf)))            # KL(real || synth)


rows = []
for label, _, _ in RUNS:
    Xr, Xf = GEN[label]["Xr"], GEN[label]["Xf"]
    F = Xr.shape[2]
    rh, fh = Xr.mean(0), Xf.mean(0)                       # (168, F) diurnal profiles
    hc = [np.corrcoef(rh[:, k], fh[:, k])[0, 1] for k in range(F)
          if rh[:, k].std() > 1e-6 and fh[:, k].std() > 1e-6]
    hc = [v for v in hc if np.isfinite(v)]
    rs = Xr.transpose(0, 2, 1).reshape(-1, 168)
    fs = Xf.transpose(0, 2, 1).reshape(-1, 168)
    rows.append(dict(
        run=label,
        n_kpi=F,
        div_ratio=float(Xf.std(0).mean() / Xr.std(0).mean()),
        std_ratio_med=float(np.median(Xf.std((0, 1)) / (Xr.std((0, 1)) + 1e-8))),
        hour_corr=float(np.median(hc)),
        ks_med=float(np.median([ks_2samp(Xr[:, :, k].ravel(), Xf[:, :, k].ravel()).statistic for k in range(F)])),
        kl_med=float(np.median([kl_hist(Xr[:, :, k].ravel(), Xf[:, :, k].ravel()) for k in range(F)])),
        ac1_gap=abs(lag_ac(fs, 1) - lag_ac(rs, 1)),
        ac24_gap=abs(lag_ac(fs, 24) - lag_ac(rs, 24)),
        loss_final=history(label)["loss"][-1],
    ))

M = pd.DataFrame(rows).set_index("run")
M.round(4)

In [ ]:
specs = [
    ("div_ratio",     "R1 dispersion (synth/real std)  →1 best", 1.0),
    ("std_ratio_med", "R1 per-KPI std ratio  →1 best",           1.0),
    ("hour_corr",     "diurnal profile corr  →1 best",           1.0),
    ("ks_med",        "R3 KS median (cross-run)  ↓ best",        None),
    ("kl_med",        "R3 marginal KL median (within-space)  ↓ best", None),
    ("ac24_gap",      "R2 |lag-24 AC gap|  ↓ best",              None),
]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(M)))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (col, title, tgt) in zip(axes.ravel(), specs):
    vals = M[col].values
    ax.bar(M.index, vals, color=colors)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis="x", rotation=20)
    if tgt is not None:
        ax.axhline(tgt, ls="--", c="k", alpha=0.5)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

## 4. Per-KPI distributional spread

Same metric, but the full per-KPI distribution rather than just the median — computed
on the **shared KPI set** (intersection of all runs) so it's like-for-like. KS (left)
is cross-run valid; KL (right, log scale) is within-space. Box = IQR, line = median,
triangle = mean.

In [ ]:
shared = sorted(set.intersection(*[set(GEN[l]["kc"]) for l, _, _ in RUNS]))
print(f"{len(shared)} shared KPIs")

ks_by, kl_by = {}, {}
for label, _, _ in RUNS:
    g = GEN[label]
    col = {k: i for i, k in enumerate(g["kc"])}
    Xr, Xf = g["Xr"], g["Xf"]
    ks_by[label] = [ks_2samp(Xr[:, :, col[k]].ravel(), Xf[:, :, col[k]].ravel()).statistic for k in shared]
    kl_by[label] = [kl_hist(Xr[:, :, col[k]].ravel(), Xf[:, :, col[k]].ravel()) for k in shared]

labels = [l for l, _, _ in RUNS]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot([ks_by[l] for l in labels], tick_labels=labels, showmeans=True)
axes[0].set_title(f"per-KPI KS (shared {len(shared)} KPIs, cross-run)  ↓"); axes[0].set_ylabel("KS statistic")
axes[1].boxplot([kl_by[l] for l in labels], tick_labels=labels, showmeans=True)
axes[1].set_title("per-KPI marginal KL (within-space)  ↓"); axes[1].set_ylabel("KL (log)"); axes[1].set_yscale("log")
plt.tight_layout(); plt.show()

## 5. Marginal overlays for the hard KPIs

`NR_1400` (skewed) is the YJ win; `NR_1266`/`NR_1225` (near-delta spikes) are the
still-open problem. Each panel overlays real vs synth **in that run's own scaled
space** — so compare real-vs-synth *within* a panel; the x-axis differs between raw
(run_3/4) and YJ (`*_yj`) columns.

In [ ]:
NAMED = [k for k in ["NR_135", "NR_1400", "NR_1266", "NR_1225"]
         if all(k in GEN[l]["kc"] for l, _, _ in RUNS)]

fig, axes = plt.subplots(len(NAMED), len(RUNS), figsize=(4 * len(RUNS), 2.8 * len(NAMED)), squeeze=False)
for r, kpi in enumerate(NAMED):
    for c, (label, _, _) in enumerate(RUNS):
        g = GEN[label]; j = g["kc"].index(kpi); ax = axes[r][c]
        ax.hist(g["Xr"][:, :, j].ravel(), bins=40, density=True, alpha=0.5, label="real")
        ax.hist(g["Xf"][:, :, j].ravel(), bins=40, density=True, alpha=0.5, label="synth")
        if r == 0:
            ax.set_title(label)
        if c == 0:
            ax.set_ylabel(kpi, fontsize=11)
        if r == 0 and c == 0:
            ax.legend(fontsize=8)
plt.tight_layout(); plt.show()